In [ ]:
import pandas as pd
import torch
import librosa
from tsai.all import *


# Load in Music Data
music_dta_3 = pd.read_csv('/Users/tarush/Desktop/STA 395/ML_Music_Genre-main/Data/music_with_labels_3_genres.csv')

# Load in base model
net_3 = RocketClassifier()

# Create additional variables 
path = '/Users/tarush/Desktop/STA 395/ML_Music_Genre-main/Data/Music/genres_original' 
music_dta_3["Audio_pth"] = path + "/" + music_dta_3["Label"] + "/" + music_dta_3["Audio"]
music_dta_3['Audio_y_dta'] = (music_dta_3['Audio_pth'].map(librosa.load))



In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn

# Grab portion of amplitude data
def grab_first(inp):
    return (inp[0])[270000:390000]

music_dta_3['Audio_y_dta_arr'] = music_dta_3['Audio_y_dta'].apply(grab_first)

# Split data to create training and testing set
music_dta_3_train, music_dta_3_test = train_test_split(music_dta_3, test_size=0.25)


# data for training
music_dta_3_train['Label_num'], unique_categories = pd.factorize(music_dta_3_train["Label"]) # Stack Overflow

music_dta_3_y_train = music_dta_3_train['Label_num'].to_numpy()

music_dta_3_X_train = music_dta_3_train['Audio_y_dta_arr'].to_numpy()
music_dta_3_X_train = np.concatenate(music_dta_3_X_train)
music_dta_3_X_train = music_dta_3_X_train.reshape(225, 1, 120000) # adjust as needed, (obs, 1, time series data length)

# data for testing
music_dta_3_test['Label_num'], unique_categories = pd.factorize(music_dta_3_test["Label"]) # Stack Overflow

music_dta_3_y_test = music_dta_3_test['Label_num'].to_numpy()

music_dta_3_X_test = music_dta_3_test['Audio_y_dta_arr'].to_numpy()
music_dta_3_X_test = np.concatenate(music_dta_3_X_test)
music_dta_3_X_test = music_dta_3_X_test.reshape(75, 1, 120000) # adjust as needed, (obs, 1, time series data length)

(14400000,)
(225, 1, 64000)
(4800000,)
(75, 1, 64000)


In [ ]:
rck_mod = RocketClassifier(num_kernels=100000) # adjust kernels as see fit

# fit model to training data
rck_mod.fit(music_dta_3_X_train, music_dta_3_y_train)

Pipeline(steps=[('rocket', Rocket(num_kernels=100000)), ('scalar', StandardScaler(with_mean=False)), ('ridgeclassifiercv', RidgeClassifierCV(alphas=array([1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03])))])

In [ ]:
# predict on testing inputs
y_pred = rck_mod.predict(music_dta_3_X_test)

In [ ]:
from sklearn.metrics import accuracy_score
# determine accuracy on testing data
print(accuracy_score(music_dta_3_y_test, y_pred))

0.16
